# Cost-Aware Reasoning Benchmark — v3

Shared engine with structured output, bounded retries, arithmetic checks, terminal review, full traces and three seeded runs. See manifest.json for comparison policy. Official dataset profiles: pilot defaults to 20 GSM8K training questions and one seed. Switch PROFILE to evaluation for 100 GSM8K test + 50 SVAMP questions and three seeds. Dataset selection uses a separate fixed seed; IDs and source hashes are exported. Includes a Thought of Search arithmetic adaptation: generate search components up front, then execute local BFS without LLM calls during search. This per-problem restricted-code adaptation omits the original human validation and domain-level reuse. [Paper](https://arxiv.org/abs/2404.11833). Results from earlier implementations must be reported separately.


In [ ]:
# Enable Internet and select a GPU in Kaggle settings before running.
!nvidia-smi
!apt-get update -y && apt-get install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama scikit-learn pandas matplotlib -q
import subprocess, urllib.request, time
try:
    urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2).close()
except Exception:
    ollama_log = open('ollama_server.log', 'a')
    ollama_process = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=ollama_log)
    for attempt in range(30):
        try:
            urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2).close()
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError('Ollama failed to start; inspect ollama_server.log')
!ollama pull qwen2.5:7b


In [ ]:
Model = 'qwen2.5:7b'


## Shared solver and validation


In [ ]:
"""Shared, traceable benchmark implementation; also embedded in the Kaggle notebook.

No solver receives the reference answer until scoring. Unfinished branches never
become answers. Arithmetic checks verify equalities, not word-problem semantics.
"""
import ast
import heapq
import itertools
import json
import math
import operator
import re
import time
from dataclasses import dataclass

VERSION = '4'
METHODS = ('Standard CoT (1-Call)', 'Step-wise Greedy CoT',
           'Tree-of-Thoughts (LLM-Judge)', 'Cost-Aware Priority Search',
           'Ablation: Quality-Only (w3=0)', 'Thought of Search (Arithmetic Adaptation)')

STEP_SCHEMA = {
    'type': 'object', 'additionalProperties': False,
    'properties': {
        'step': {'type': 'string'}, 'equation': {'type': 'string'},
        'done': {'type': 'boolean'}, 'answer': {'type': ['number', 'null']},
        'target': {'type': 'string'},
        'expression': {'type': 'string'}, 'result': {'type': ['number', 'null']},
    }, 'required': ['step', 'equation', 'done', 'answer', 'target'],
}
REVIEW_SCHEMA = {
    'type': 'object', 'additionalProperties': False,
    'properties': {'valid': {'type': 'boolean'}, 'answer': {'type': 'number'},
                   'reason': {'type': 'string'}},
    'required': ['valid', 'answer', 'reason'],
}
JUDGE_SCHEMA = {
    'type': 'object', 'additionalProperties': False,
    'properties': {'score': {'type': 'number', 'minimum': 0, 'maximum': 1}},
    'required': ['score'],
}


@dataclass(frozen=True)
class Config:
    seed: int = 11
    max_depth: int = 10
    beam: int = 3
    candidates: int = 2
    max_expansions: int = 40
    max_calls: int = 240
    retries: int = 2
    step_tokens: int = 512
    final_tokens: int = 1536
    max_tokens: int = 4096
    w1: float = 0.5
    audit_tos: bool = False
    review_appeal: bool = False

    def __post_init__(self):
        for key in ('max_depth', 'beam', 'candidates', 'max_expansions', 'max_calls',
                    'step_tokens', 'final_tokens', 'max_tokens'):
            if getattr(self, key) < 1:
                raise ValueError(key + ' must be positive')
        if self.retries < 0 or not 0 <= self.w1 <= 1:
            raise ValueError('invalid retries or w1')


def numeric(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)


def safe_arithmetic(expr):
    """Small arithmetic language; reject prose and bound expensive operations."""
    if not isinstance(expr, str) or len(expr) > 256:
        return None
    ops = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
           ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv,
           ast.Mod: operator.mod, ast.Pow: operator.pow}

    def evaluate(node):
        if isinstance(node, ast.Constant) and numeric(node.value):
            value = node.value
        elif isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.USub, ast.UAdd)):
            value = evaluate(node.operand) * (-1 if isinstance(node.op, ast.USub) else 1)
        elif isinstance(node, ast.BinOp) and type(node.op) in ops:
            left, right = evaluate(node.left), evaluate(node.right)
            if isinstance(node.op, ast.Pow) and abs(right) > 12:
                raise ValueError('exponent limit')
            value = ops[type(node.op)](left, right)
        else:
            raise ValueError('unsupported arithmetic')
        if not numeric(value) or abs(value) > 1e15:
            raise ValueError('arithmetic limit')
        return value

    try:
        tree = ast.parse(expr.strip().replace('$', '').replace(',', ''), mode='eval')
        if sum(1 for _ in ast.walk(tree)) > 80:
            return None
        return float(evaluate(tree.body))
    except (ValueError, SyntaxError, ArithmeticError, TypeError, RecursionError):
        return None


def equation_status(equation):
    if not equation or equation.lower() == 'none':
        return 'missing', None
    if equation.count('=') != 1:
        return 'unsupported', None
    lhs, rhs = map(safe_arithmetic, equation.split('='))
    if lhs is None or rhs is None:
        return 'unsupported', None
    return ('valid' if math.isclose(lhs, rhs, rel_tol=1e-9, abs_tol=1e-6) else 'invalid'), rhs


def equation_key(equation):
    try:
        return '|'.join(ast.dump(ast.parse(s.strip().replace('$', '').replace(',', ''), mode='eval'))
                        for s in equation.split('='))
    except SyntaxError:
        return equation.lower().replace(' ', '')


def parse_object(raw):
    text = raw.strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text)
    obj = json.loads(text)
    if not isinstance(obj, dict):
        raise ValueError('expected one JSON object')
    return obj


def validate_step(obj):
    if not set(STEP_SCHEMA['required']).issubset(obj) or not set(obj).issubset(STEP_SCHEMA['properties']):
        raise ValueError('missing or unexpected fields')
    if any(not isinstance(obj[k], str) for k in ('step', 'equation', 'target')):
        raise ValueError('text fields must be strings')
    if not isinstance(obj['done'], bool):
        raise ValueError('invalid done flag')
    expression = obj.get('expression', '').strip()
    result = obj.get('result')
    if expression or result is not None:
        computed = safe_arithmetic(expression)
        if computed is None or not numeric(result) or not math.isclose(
                computed, result, rel_tol=1e-9, abs_tol=1e-6):
            raise ValueError('expression_result_mismatch')
        obj = dict(obj, equation=f'{expression} = {result}')
    if not obj['step'].strip():
        if equation_status(obj['equation'])[0] != 'valid' or not obj['target'].strip():
            raise ValueError('empty step without verified calculation')
        obj = dict(obj, step='Computed ' + obj['target'].strip())
    if obj['done']:
        if not numeric(obj['answer']) or not obj['target'].strip():
            raise ValueError('completed answer needs a number and target')
    elif obj['answer'] is not None:
        raise ValueError('unfinished step must have null answer')
    # A chain is acceptable only when EVERY expression has the same numeric value.
    # Keep the original response in the request trace; normalize the validated object.
    if obj['equation'].count('=') > 1:
        parts = obj['equation'].split('=')
        if len(parts) > 8 or len(obj['equation']) > 512:
            raise ValueError('equation_chain_limit')
        values = [safe_arithmetic(part) for part in parts]
        if any(value is None for value in values):
            raise ValueError('equation_unsupported')
        if not all(math.isclose(values[0], value, rel_tol=1e-9, abs_tol=1e-6) for value in values[1:]):
            raise ValueError('equation_invalid')
        obj = dict(obj, equation=parts[0].strip() + ' = ' + parts[-1].strip())
    status, _ = equation_status(obj['equation'])
    if status in ('unsupported', 'invalid'):
        raise ValueError('equation_' + status)
    if not obj['done'] and status != 'valid':
        raise ValueError('arithmetic step needs a valid equation')
    return obj


def extract_explicit_answer(text):
    """Only a complete final marker; never guess from intermediate numbers."""
    matches = re.findall(r'^\s*(?:\*\*)?FINAL ANSWER:\s*(?:\*\*)?\s*\$?\s*'
                         r'([+-]?(?:\d[\d,]*(?:\.\d+)?|\.\d+))\s*(?:\*\*)?\s*$',
                         text, re.MULTILINE | re.IGNORECASE)
    if len(matches) != 1:
        return None
    value = float(matches[0].replace(',', ''))
    return value if numeric(value) else None


class Solver:
    """backend(prompt, schema, temperature, seed, max_tokens) -> response dict.

    Response keys: text, prompt_tokens, completion_tokens, done_reason.
    All real requests including retries and terminal review count toward cost.
    """
    def __init__(self, backend, config=None):
        self.backend = backend
        self.cfg = config or Config()
        self.events = []
        self.calls = self.tokens = 0
        self.failure = None
        self.rejections = {}
        self.review_cache = {}

    def request(self, prompt, schema, temp, purpose, limit):
        feedback = ''
        for attempt in range(self.cfg.retries + 1):
            if self.calls >= self.cfg.max_calls:
                self.failure = 'call_limit'
                return None
            seed = (self.cfg.seed + self.calls) % (2**31)
            self.calls += 1
            event = {'kind': 'request', 'purpose': purpose, 'attempt': attempt,
                     'seed': seed, 'prompt': prompt + feedback, 'token_limit': limit,
                     'schema': schema, 'temperature': temp}
            self.events.append(event)
            try:
                response = self.backend(prompt + feedback, schema, temp, seed, limit)
                event.update(response)
                self.tokens += response.get('prompt_tokens', 0) + response.get('completion_tokens', 0)
                if response.get('done_reason') == 'length':
                    raise ValueError('truncated')
                raw = response['text']
                obj = parse_object(raw) if schema else extract_explicit_answer(raw)
                if schema == STEP_SCHEMA:
                    obj = validate_step(obj)
                elif schema == REVIEW_SCHEMA:
                    if (not isinstance(obj.get('valid'), bool) or not numeric(obj.get('answer'))
                            or not isinstance(obj.get('reason'), str)):
                        raise ValueError('invalid review')
                elif schema == JUDGE_SCHEMA:
                    if not numeric(obj.get('score')):
                        raise ValueError('invalid judge score')
                    score = obj['score']
                    if -1e-9 <= score <= 1 + 1e-9 and not 0 <= score <= 1:
                        obj['score'] = min(1.0, max(0.0, score))
                        event['score_clamped_from'] = score
                    elif not 0 <= score <= 1:
                        raise ValueError('invalid judge score')
                if obj is None:
                    raise ValueError('missing_final_answer')
                event['parsed'] = obj
                self.failure = None
                return obj
            except Exception as exc:
                event['error'] = str(exc)
                self.failure = str(exc)
                instruction = (
                    'Return the requested JSON search model with valid function definitions.'
                    if purpose == 'tos_model_synthesis' else
                    'Return calculations, result and target in the executable-equations JSON schema.'
                    if purpose == 'executable_equations' else
                    'Return valid (boolean), answer (number), and reason (string).'
                    if schema == REVIEW_SCHEMA else
                    'Return a score from zero to one in the JSON schema.'
                    if schema == JUDGE_SCHEMA else
                    'Return step, equation, done, answer and target. Include expression and numeric result '
                    'as separate fields; their arithmetic must agree. Use a numeric arithmetic equality; '
                    'done is a boolean and answer is null until the final quantity is solved.'
                    if schema == STEP_SCHEMA else
                    'Finish with FINAL ANSWER: <decimal number> on its own line.')
                feedback = ('\nPrevious response: ' + str(event.get('text', ''))[:2000]
                            + '\nYour previous response was rejected: ' + str(exc) + '. ' + instruction)
                if str(exc) == 'truncated':
                    limit = min(limit * 2, self.cfg.max_tokens)
        return None

    def terminal_review(self, question, steps, candidate):
        # No expected answer enters this prompt. This common check costs real tokens.
        key = (tuple(equation_key(step['equation']) for step in steps + [candidate]),
               candidate['answer'], candidate['target'].strip().lower())
        if key in self.review_cache:
            self.events.append({'kind': 'terminal_cache_hit', 'candidate': candidate})
            return self.review_cache[key]
        prompt = (f'Independently solve this problem: {question}\n'
                  f'Proposed reasoning: {json.dumps(steps + [candidate])}\n'
                  'Check the requested quantity, units, use of givens, and complete reasoning. '
                  'An intermediate subtotal, speed instead of distance, or number copied from '
                  'the question is insufficient. Return valid=true only if the proposed numeric '
                  'answer and target solve the whole problem. Supply your independently computed '
                  'answer and explain your decision. Do not merely check the last equation.')
        review = self.request(prompt, REVIEW_SCHEMA, 0.1, 'terminal_review', self.cfg.final_tokens)
        accepted = (review is not None and review['valid'] and
                    math.isclose(review['answer'], candidate['answer'], rel_tol=0, abs_tol=1e-4))
        if self.cfg.review_appeal and not accepted and self.calls < self.cfg.max_calls:
            # One independent appeal can recover a false negative. Both votes are logged.
            appeal = self.request(prompt + '\nRe-evaluate independently. Check every quantity '
                                  'and the final arithmetic before deciding.',
                                  REVIEW_SCHEMA, 0.1, 'terminal_review_appeal', self.cfg.final_tokens)
            accepted = (appeal is not None and appeal['valid'] and
                        math.isclose(appeal['answer'], candidate['answer'], rel_tol=0, abs_tol=1e-4))
        else:
            appeal = None
        self.events.append({'kind': 'terminal_decision', 'candidate': candidate,
                            'review': review, 'appeal': appeal, 'accepted': accepted})
        self.review_cache[key] = accepted
        return accepted

    def solve(self, question, method):
        if method == METHODS[5]:
            return solve_thought_of_search(self, question)
        started = time.time()
        answer, status, best = None, 'exhausted', []
        if method == METHODS[0]:
            # Exactly one baseline request: truncation is diagnosed, never silently retried.
            from dataclasses import replace
            previous = self.cfg
            self.cfg = replace(previous, retries=0)
            answer = self.request(
                f'Problem: {question}\nSolve fully with clear reasoning. Finish on a new line '
                'with exactly FINAL ANSWER: <number>. Use a decimal number without units '
                'on that line.', None, 0.1, 'standard_cot', self.cfg.max_tokens)
            self.cfg = previous
            status = 'done' if answer is not None else (self.failure or 'missing_final_answer')
        else:
            greedy = method == METHODS[1]
            judge = method == METHODS[2]
            width = 1 if greedy else self.cfg.beam
            n = 1 if greedy else self.cfg.candidates
            w3 = 0.1 if method == METHODS[3] else 0.0
            counter = itertools.count()
            heap = [(-1., next(counter), [])]
            seen_paths = set()
            best_quality = -1.
            expansions = children = 0
            stalled = {}
            while heap and expansions < self.cfg.max_expansions and self.calls < self.cfg.max_calls:
                neg, _, steps = heapq.heappop(heap)
                if len(steps) >= self.cfg.max_depth:
                    continue
                expansions += 1
                self.events.append({'kind': 'expand', 'priority': -neg, 'steps': steps,
                                    'depth': len(steps), 'expansion': expansions})
                history = '\n'.join(s['target'] + ': ' + s['equation'] for s in steps)
                prompt = (f'Problem: {question}\nSteps completed: {history}\n'
                          'Give ONE new arithmetic step. Use the step schema. equation must '
                          'contain only numbers, parentheses, arithmetic operators and one =, '
                          'e.g. 3 * 12 = 36. No units, variables or prose in equation. '
                          'Use // and arithmetic to express rounding when needed. '
                          'Do not repeat a previous calculation. done is a boolean. '
                          'Until the entire requested quantity is solved, answer=null and done=false. '
                          'When solved, done=true, answer is the numeric final answer, target '
                          'describes the requested quantity and units; equation may be empty '
                          'if an earlier step already calculated that answer. Set done=true in the '
                          'same response that computes the requested final quantity. Explain the result in step. '
                          'Include expression as a numeric arithmetic expression and result as its numeric value '
                          'for every computed step.')
                branch_key = json.dumps(steps, sort_keys=True)
                if branch_key in stalled:
                    prompt += '\nThis branch stalled: ' + stalled[branch_key]['reason']
                    prompt += '. Produce a valid new calculation, or mark the computed final answer done=true.'
                if branch_key in self.rejections:
                    prompt += '\nEarlier terminal answers were rejected: ' + json.dumps(self.rejections[branch_key])
                    prompt += '. Recheck the requested quantity and continue with a new calculation.'
                candidates = []
                for _ in range(n):
                    c = self.request(prompt, STEP_SCHEMA, 0.1 if greedy else 0.7,
                                     'generate', self.cfg.step_tokens)
                    if c is None:
                        continue
                    eq_state, rhs = equation_status(c['equation'])
                    repeated = not c['done'] and any(
                        c['step'].strip().lower() == s['step'].strip().lower() or
                        equation_key(c['equation']) == equation_key(s['equation'])
                        for s in steps)
                    if repeated or c in candidates:
                        self.events.append({'kind': 'rejected', 'reason': 'repeated_step', 'candidate': c})
                        # One bounded replacement request with explicit feedback.
                        c = self.request(prompt + '\nThe previous candidate repeated a calculation. '
                                         'Advance to a new calculation or declare the final answer.',
                                         STEP_SCHEMA, 0.7, 'repeat_recovery', self.cfg.step_tokens)
                        if c is None or c in candidates or (not c['done'] and any(
                            c['step'].strip().lower() == s['step'].strip().lower() or
                            equation_key(c['equation']) == equation_key(s['equation'])
                            for s in steps)):
                            continue
                    candidates.append(c)
                if not candidates:
                    attempt = stalled.get(branch_key, {}).get('attempts', 0)
                    reason = self.failure or 'all candidates repeated earlier calculations'
                    self.events.append({'kind': 'branch_stalled', 'reason': reason, 'steps': steps})
                    if attempt < self.cfg.retries:
                        stalled[branch_key] = {'attempts': attempt + 1, 'reason': reason}
                        heapq.heappush(heap, (neg, next(counter), steps))
                children += len(candidates)
                texts = [c['step'] + ' ' + c['target'] for c in candidates]
                relevance = lexical_relevance(texts, question)
                completed = []
                for c, t in zip(candidates, relevance):
                    eq_state, rhs = equation_status(c['equation'])
                    r = 1.0 if eq_state == 'valid' else 0.25
                    observed_output = [e['completion_tokens'] for e in self.events
                                       if e.get('purpose') == 'generate'
                                       and isinstance(e.get('completion_tokens'), (int, float))]
                    expected_output = (sum(observed_output) / len(observed_output)
                                       if observed_output else 192)
                    next_history = prompt + '\n' + c['target'] + ': ' + c['equation']
                    next_prompt = len(next_history.encode('utf-8')) / 3
                    cost = next_prompt + expected_output
                    score = r * (self.cfg.w1 + (1-self.cfg.w1)*t) / (1+w3*cost/500)
                    if judge:
                        rating = self.request(
                            f'Problem: {question}\nPrior steps: {json.dumps(steps)}\n'
                            f'Candidate: {json.dumps(c)}\nRate progress and correctness from 0 to 1.',
                            JUDGE_SCHEMA, 0.1, 'judge', self.cfg.step_tokens)
                        if rating is None:
                            continue
                        score = rating['score']
                    self.events.append({'kind': 'candidate_score', 'candidate': c, 'depth': len(steps),
                                        'equation_status': eq_state, 'r': r, 't': t,
                                        'cost': cost, 'score': score})
                    if c['done']:
                        # Final numeric answer must be grounded in a verified calculation.
                        values = [equation_status(s['equation'])[1] for s in steps + [c]]
                        grounded = any(v is not None and math.isclose(v, c['answer'], abs_tol=1e-4,
                                                                    rel_tol=0) for v in values)
                        if grounded and self.terminal_review(question, steps, c):
                            completed.append((score, c, steps + [c]))
                        else:
                            self.events.append({'kind': 'rejected', 'reason': 'terminal_unverified',
                                                'candidate': c, 'grounded': grounded})
                            reasons = self.rejections.setdefault(branch_key, [])
                            reasons.append(c)
                            # Revisit this branch at most twice; global budgets still apply.
                            if len(reasons) <= self.cfg.retries:
                                heapq.heappush(heap, (-score, next(counter), steps))
                        continue
                    path = steps + [c]
                    if score > best_quality:
                        best_quality, best = score, path
                    signature = tuple(equation_key(s['equation']) for s in path)
                    if signature not in seen_paths:
                        seen_paths.add(signature)
                        heapq.heappush(heap, (-score, next(counter), path))
                if completed:
                    _, chosen, best = max(completed, key=lambda item: item[0])
                    answer, status = chosen['answer'], 'done'
                    break
                heap = heapq.nsmallest(width, heap)
                heapq.heapify(heap)
            if answer is None:
                status = ('call_limit' if self.calls >= self.cfg.max_calls else
                          'expansion_limit' if expansions >= self.cfg.max_expansions else
                          'depth_limit' if best and len(best) >= self.cfg.max_depth else 'exhausted')
        return {'method': method, 'ans': answer, 'status': status, 'steps': best,
                'calls': self.calls, 'tokens': self.tokens, 'sec': time.time()-started,
                'trace': self.events, 'last_error': self.failure, 'seed': self.cfg.seed,
                'config': vars(self.cfg), 'engine_version': VERSION}


def lexical_relevance(texts, question):
    if not texts:
        return []
    # Smooth IDF and L2 normalization, equivalent to the original vectorizer's
    # default weighting, using only stdlib so offline regressions need no model.
    from collections import Counter
    docs = [Counter(re.findall(r'\b\w\w+\b', text.lower())) for text in texts + [question]]
    frequency = Counter(word for doc in docs for word in doc)
    vectors = []
    for doc in docs:
        weighted = {word: count * (1 + math.log((1+len(docs))/(1+frequency[word])))
                    for word, count in doc.items()}
        norm = math.sqrt(sum(weight**2 for weight in weighted.values())) or 1
        vectors.append({word: weight/norm for word, weight in weighted.items()})
    goal = vectors[-1]
    scores = [sum(weight * goal.get(word, 0) for word, weight in vec.items()) for vec in vectors[:-1]]
    lo, hi = min(scores), max(scores)
    return [(s-lo)/(hi-lo) for s in scores] if hi-lo > 1e-9 else [0.5]*len(texts)


def evaluate(backend, question, expected, method, config=None, problem_id=None):
    if method not in METHODS:
        raise ValueError('unknown method')
    result = Solver(backend, config).solve(question, method)
    ans = result['ans']
    correct = ans is not None and math.isclose(ans, expected, rel_tol=0, abs_tol=1e-4)
    result.update(question=question, expected=expected, correct=correct, problem_id=problem_id,
                  outcome='missing' if ans is None else 'correct' if correct else 'wrong',
                  final_answer=ans, expected_answer=expected)
    return result


## Thought of Search: model synthesis and local BFS


In [ ]:
"""Thought of Search arithmetic adaptation: synthesize components, then local BFS.

Reference: Katz et al., https://arxiv.org/abs/2404.11833.
The original human-reviewed, reusable domain code is replaced here by bounded
per-problem synthesis of restricted Python functions. No paper-level soundness
or completeness guarantee is claimed for the inferred word-problem model.
"""
import ast
import copy
from collections import deque
import json
import math
import operator
import time


TOS_METHOD = 'Thought of Search (Arithmetic Adaptation)'
TOS_SCHEMA = {
    'type': 'object', 'additionalProperties': False,
    'properties': {
        'initial_state': {'type': 'object', 'additionalProperties': {'type': 'number'}},
        'successor_code': {'type': 'string'}, 'goal_code': {'type': 'string'},
        'answer_code': {'type': 'string'}, 'target': {'type': 'string'},
        'objective': {'type': 'string', 'enum': ['first', 'min', 'max']},
    },
    'required': ['initial_state', 'successor_code', 'goal_code', 'answer_code', 'target', 'objective'],
}


class RestrictedFunction:
    """Interpret bounded arithmetic Python; never exec/eval generated code."""
    allowed = (ast.Expression, ast.Constant, ast.Name, ast.Load, ast.Store,
               ast.Subscript, ast.Dict, ast.List, ast.Tuple, ast.BinOp, ast.UnaryOp,
               ast.BoolOp, ast.Compare, ast.IfExp, ast.ListComp, ast.comprehension,
               ast.Call, ast.Add, ast.Sub, ast.Mult, ast.Div, ast.FloorDiv, ast.Mod,
               ast.Pow, ast.USub, ast.UAdd, ast.Not, ast.And, ast.Or, ast.Eq,
               ast.NotEq, ast.Lt, ast.LtE, ast.Gt, ast.GtE, ast.BitOr,
               ast.Assign, ast.AugAssign, ast.If, ast.For, ast.Return, ast.Expr,
               ast.Break, ast.Continue, ast.Pass, ast.Attribute, ast.keyword)
    functions = {'abs', 'min', 'max', 'sum', 'len', 'round', 'ceil', 'floor', 'range',
                 'dict', 'int', 'float'}
    methods = {'copy', 'get', 'update', 'append'}

    def __init__(self, code, name):
        if not isinstance(code, str) or len(code) > 12000:
            raise ValueError('component code must be a string of at most 12000 characters')
        tree = ast.parse(code)
        if len(tree.body) != 1 or not isinstance(tree.body[0], ast.FunctionDef):
            raise ValueError('component must define exactly one function')
        function = tree.body[0]
        args = function.args
        if (function.name != name or function.decorator_list or function.returns
                or len(args.args) != 1 or args.args[0].arg != 'state' or args.args[0].annotation
                or args.posonlyargs or args.kwonlyargs or args.vararg or args.kwarg or args.defaults
                or args.kw_defaults):
            raise ValueError('use def ' + name + '(state): without decorators, defaults, or annotations')
        self.body = function.body
        nodes = [node for statement in self.body for node in ast.walk(statement)]
        if not any(isinstance(node, ast.Return) for node in nodes):
            raise ValueError('function requires a return statement')
        if len(nodes) > 1000:
            raise ValueError('component AST is too large')
        for node in nodes:
            if not isinstance(node, self.allowed):
                raise ValueError('unsupported syntax: ' + type(node).__name__)
            if isinstance(node, ast.Name) and node.id.startswith('_'):
                raise ValueError('private names are unsupported')
            if isinstance(node, ast.Attribute) and node.attr not in self.methods:
                raise ValueError('unsupported method or attribute: ' + node.attr)
            if isinstance(node, ast.Call):
                helper = isinstance(node.func, ast.Name) and node.func.id in self.functions
                method = isinstance(node.func, ast.Attribute) and node.func.attr in self.methods
                if not (helper or method) or any(isinstance(arg, ast.Starred) for arg in node.args):
                    raise ValueError('only arithmetic helpers and bounded collection methods are supported')
                if any(keyword.arg is None for keyword in node.keywords):
                    raise ValueError('keyword unpacking is unsupported')
                if node.keywords and not (helper and node.func.id == 'dict'):
                    raise ValueError('only dict accepts keyword arguments')
            if isinstance(node, ast.comprehension) and (
                    node.is_async or not isinstance(node.target, ast.Name)
                    or node.target.id == 'state' or node.target.id in self.functions):
                raise ValueError('comprehensions need a simple local variable')
        if not any(isinstance(node, ast.Name) and node.id == 'state' for node in nodes):
            raise ValueError('component must depend on state, not a constant answer')

    def __call__(self, state):
        self.remaining = 20000
        # Generated assignments cannot mutate states retained in the BFS queue or trace.
        outcome = self.execute(self.body, {'state': copy.deepcopy(state)})
        if outcome is None or outcome[0] != 'return':
            raise ValueError('function reached its end without returning')
        return outcome[1]

    def tick(self):
        self.remaining -= 1
        if self.remaining < 0:
            raise ValueError('expression operation limit')

    def assign(self, target, value, env):
        if isinstance(target, ast.Name) and target.id not in self.functions and not target.id.startswith('_'):
            env[target.id] = value
        elif isinstance(target, ast.Subscript):
            collection = self.visit(target.value, env)
            key = self.visit(target.slice, env)
            if not isinstance(collection, (dict, list)):
                raise ValueError('assignment needs a local collection')
            collection[key] = value
            if len(collection) > 256:
                raise ValueError('collection limit')
        else:
            raise ValueError('unsupported assignment target')

    def execute(self, statements, env):
        for statement in statements:
            self.tick()
            if isinstance(statement, ast.Return):
                return 'return', self.visit(statement.value, env) if statement.value else None
            if isinstance(statement, ast.Assign):
                value = self.visit(statement.value, env)
                for target in statement.targets:
                    self.assign(target, value, env)
            elif isinstance(statement, ast.AugAssign):
                expression = ast.BinOp(left=statement.target, op=statement.op, right=statement.value)
                self.assign(statement.target, self.visit(expression, env), env)
            elif isinstance(statement, ast.If):
                branch = statement.body if self.visit(statement.test, env) else statement.orelse
                outcome = self.execute(branch, env)
                if outcome:
                    return outcome
            elif isinstance(statement, ast.For):
                values = self.visit(statement.iter, env)
                if not isinstance(values, list) or len(values) > 256:
                    raise ValueError('for requires a bounded list or range')
                broke = False
                for value in list(values):
                    self.tick()
                    self.assign(statement.target, value, env)
                    outcome = self.execute(statement.body, env)
                    if outcome and outcome[0] == 'return':
                        return outcome
                    if outcome and outcome[0] == 'break':
                        broke = True
                        break
                if not broke:
                    outcome = self.execute(statement.orelse, env)
                    if outcome:
                        return outcome
            elif isinstance(statement, ast.Break):
                return 'break', None
            elif isinstance(statement, ast.Continue):
                return 'continue', None
            elif isinstance(statement, ast.Expr):
                self.visit(statement.value, env)
            elif not isinstance(statement, ast.Pass):
                raise ValueError('unsupported statement')

    def visit(self, node, env):
        self.tick()
        if isinstance(node, ast.Constant):
            value = node.value
            if not isinstance(value, (str, bool)) and not numeric(value):
                raise ValueError('unsupported constant')
            if isinstance(value, str) and len(value) > 64:
                raise ValueError('string limit')
        elif isinstance(node, ast.Name):
            if node.id not in env:
                raise ValueError('undefined local variable: ' + node.id)
            value = env[node.id]
        elif isinstance(node, ast.Subscript):
            collection, key = self.visit(node.value, env), self.visit(node.slice, env)
            try:
                value = collection[key]
            except KeyError as exc:
                raise ValueError(f'Undefined state key {key!r}; dictionary literals read the OLD state. '
                                 'Compute dependent fields with sequential assignments or separate stages.') from exc
        elif isinstance(node, ast.Dict):
            if len(node.keys) > 32:
                raise ValueError('state size limit')
            value = {}
            for key, child in zip(node.keys, node.values):
                if key is None:
                    mapping = self.visit(child, env)
                    if not isinstance(mapping, dict):
                        raise ValueError('dictionary unpacking needs a dictionary')
                    value.update(mapping)
                else:
                    value[self.visit(key, env)] = self.visit(child, env)
        elif isinstance(node, (ast.List, ast.Tuple)):
            if len(node.elts) > 256:
                raise ValueError('collection limit')
            value = [self.visit(item, env) for item in node.elts]
        elif isinstance(node, ast.BinOp):
            a, b = self.visit(node.left, env), self.visit(node.right, env)
            if isinstance(node.op, ast.BitOr):
                if not isinstance(a, dict) or not isinstance(b, dict):
                    raise ValueError('union is supported only for dictionaries')
                value = dict(a, **b)
                if len(value) > 32:
                    raise ValueError('state size limit')
                return value
            if not numeric(a) or not numeric(b):
                raise ValueError('arithmetic requires numbers')
            if isinstance(node.op, ast.Pow) and abs(b) > 12:
                raise ValueError('exponent limit')
            operations = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
                          ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv,
                          ast.Mod: operator.mod, ast.Pow: operator.pow}
            value = operations[type(node.op)](a, b)
        elif isinstance(node, ast.UnaryOp):
            a = self.visit(node.operand, env)
            if isinstance(node.op, ast.Not):
                value = not a
            elif numeric(a):
                value = -a if isinstance(node.op, ast.USub) else a
            else:
                raise ValueError('unary arithmetic requires a number')
        elif isinstance(node, ast.BoolOp):
            value = self.visit(node.values[0], env)
            for child in node.values[1:]:
                if (isinstance(node.op, ast.And) and not value) or (isinstance(node.op, ast.Or) and value):
                    break
                value = self.visit(child, env)
        elif isinstance(node, ast.Compare):
            a = self.visit(node.left, env)
            operations = {ast.Eq: operator.eq, ast.NotEq: operator.ne, ast.Lt: operator.lt,
                          ast.LtE: operator.le, ast.Gt: operator.gt, ast.GtE: operator.ge}
            value = True
            for op, child in zip(node.ops, node.comparators):
                b = self.visit(child, env)
                if not operations[type(op)](a, b):
                    value = False
                    break
                a = b
        elif isinstance(node, ast.IfExp):
            value = self.visit(node.body if self.visit(node.test, env) else node.orelse, env)
        elif isinstance(node, ast.ListComp):
            value = []
            def collect(index, scope):
                if index == len(node.generators):
                    value.append(self.visit(node.elt, scope))
                    if len(value) > 256:
                        raise ValueError('collection limit')
                    return
                gen = node.generators[index]
                iterable = self.visit(gen.iter, scope)
                if not isinstance(iterable, list) or len(iterable) > 256:
                    raise ValueError('bounded list required')
                for item in iterable:
                    self.remaining -= 1
                    if self.remaining < 0:
                        raise ValueError('expression operation limit')
                    local = dict(scope, **{gen.target.id: item})
                    if all(self.visit(condition, local) for condition in gen.ifs):
                        collect(index + 1, local)
            collect(0, env)
        elif isinstance(node, ast.Call):
            args = [self.visit(arg, env) for arg in node.args]
            if isinstance(node.func, ast.Attribute):
                receiver = self.visit(node.func.value, env)
                method = node.func.attr
                if isinstance(receiver, dict) and method == 'copy' and not args:
                    return dict(receiver)
                if isinstance(receiver, dict) and method == 'get' and 1 <= len(args) <= 2:
                    return receiver.get(*args)
                if isinstance(receiver, dict) and method == 'update' and len(args) == 1 and isinstance(args[0], dict):
                    receiver.update(args[0])
                elif isinstance(receiver, list) and method == 'append' and len(args) == 1:
                    receiver.append(args[0])
                else:
                    raise ValueError('unsupported collection method arguments')
                if len(receiver) > 256:
                    raise ValueError('collection limit')
                return None
            name = node.func.id
            if name == 'range':
                if not 1 <= len(args) <= 3 or any(not numeric(arg) or int(arg) != arg for arg in args):
                    raise ValueError('range needs integer bounds')
                sequence = range(*(int(arg) for arg in args))
                if len(sequence) > 256:
                    raise ValueError('range limit')
                value = list(sequence)
            elif name == 'dict':
                if len(args) > 1 or (args and not isinstance(args[0], dict)):
                    raise ValueError('dict needs at most one dictionary')
                value = dict(args[0]) if args else {}
                value.update({keyword.arg: self.visit(keyword.value, env) for keyword in node.keywords})
            else:
                funcs = {'abs': abs, 'min': min, 'max': max, 'sum': sum, 'len': len,
                         'round': round, 'ceil': math.ceil, 'floor': math.floor, 'int': int, 'float': float}
                value = funcs[name](*args)
        else:
            raise ValueError('unsupported expression')
        if isinstance(value, (dict, list, tuple)) and len(value) > 256:
            raise ValueError('collection limit')
        if isinstance(value, (int, float)) and not isinstance(value, bool):
            if not numeric(value) or abs(value) > 1e15:
                raise ValueError('numeric limit')
        return value


def validate_state(state, keys=None):
    if (not isinstance(state, dict) or not 1 <= len(state) <= 32
            or any(not isinstance(key, str) or not key or len(key) > 64 for key in state)
            or any(not numeric(value) or abs(value) > 1e15 for value in state.values())):
        raise ValueError('state must have string keys and finite numeric values')
    return state


def validate_model(model):
    if set(model) != set(TOS_SCHEMA['required']):
        raise ValueError('missing or unexpected search-model fields')
    initial = validate_state(model['initial_state'])
    if model['objective'] not in ('first', 'min', 'max') or not isinstance(model['target'], str):
        raise ValueError('invalid objective or target')
    if not model['target'].strip():
        raise ValueError('target is required')
    successor = RestrictedFunction(model['successor_code'], 'successors')
    goal = RestrictedFunction(model['goal_code'], 'is_goal')
    answer = RestrictedFunction(model['answer_code'], 'answer')
    initial_goal = goal(initial)
    if not isinstance(initial_goal, bool):
        raise ValueError('goal test must return a boolean')
    children = [] if initial_goal else successor(initial)
    if not isinstance(children, list) or len(children) > 256:
        raise ValueError('initial successors must be a bounded list')
    for child in children:
        validate_state(child, set(initial))
    return initial, successor, goal, answer


def local_bfs(model, config, events):
    initial, successors, is_goal, answer = validate_model(model)
    queue = deque([(initial, [])])
    visited = {json.dumps(initial, sort_keys=True)}
    expansions = 0
    best = None
    depth_cutoff = False
    while queue:
        state, path = queue.popleft()
        goal = is_goal(state)
        if not isinstance(goal, bool):
            raise ValueError('goal test must return a boolean')
        events.append({'kind': 'tos_visit', 'state': state, 'depth': len(path), 'goal': goal})
        if goal:
            value = answer(state)
            if not numeric(value):
                raise ValueError('answer function must return a finite number')
            candidate = (value, path)
            if model['objective'] == 'first':
                return value, path, 'done', expansions
            if best is None or (value < best[0] if model['objective'] == 'min' else value > best[0]):
                best = candidate
            continue
        if len(path) >= config.max_depth:
            depth_cutoff = True
            continue
        if expansions >= config.max_expansions:
            return None, [], 'expansion_limit', expansions
        expansions += 1
        events.append({'kind': 'tos_expand', 'state': state, 'depth': len(path)})
        children = successors(state)
        if not isinstance(children, list) or len(children) > 256:
            raise ValueError('successor function must return a bounded list')
        for child in children:
            validate_state(child, set(initial))
            key = json.dumps(child, sort_keys=True)
            if key not in visited:
                visited.add(key)
                queue.append((child, path + [{'state': child}]))
        if len(visited) > config.max_expansions * 256 + 1:
            raise ValueError('state budget exceeded')
    if depth_cutoff:
        return None, [], 'depth_limit', expansions
    if best is not None:
        return best[0], best[1], 'done', expansions
    return None, [], 'exhausted', expansions


def solve_thought_of_search(solver, question):
    started = time.time()
    example = {
        'initial_state': {'bags': 2, 'each': 5, 'removed': 3, 'stage': 0, 'value': 0},
        'successor_code': (
            'def successors(state):\n'
            '    return [{"bags":state["bags"], "each":state["each"], '
            '"removed":state["removed"], "stage":state["stage"]+1, '
            '"value":state["bags"]*state["each"] if state["stage"]==0 '
            'else state["value"]-state["removed"]}] if state["stage"]<2 else []'),
        'goal_code': 'def is_goal(state):\n    return state["stage"] == 2',
        'answer_code': 'def answer(state):\n    return state["value"]',
        'target': 'beads remaining', 'objective': 'first',
    }
    prompt = (
        f'Problem: {question}\nImplement a symbolic arithmetic search model BEFORE searching. '
        'Return the JSON search-model schema. initial_state contains ONLY given numeric facts '
        'and numeric counters. States may add or omit keys, but preserve facts needed by later steps. '
        'target describes the requested quantity and units. successor_code defines def successors(state) '
        'returning a list of successor state dictionaries. goal_code defines def is_goal(state) '
        'returning a boolean. answer_code defines def answer(state) extracting the computed answer. '
        'Use ordinary local assignments, if/elif/else, bounded for loops, return, arithmetic, '
        'comparisons, conditional expressions and list comprehensions. Helpers: range, abs, min, '
        'max, sum, len, round, ceil, floor, int, float, dict. Methods: dict.copy/get/update and '
        'list.append. Dictionary union (state | updates) and {**state, ...} are supported. '
        'No imports, while loops, recursion, arbitrary attributes or external calls. '
        'range bounds must be integral, with at most 256 values. '
        'Do NOT put a guessed final answer in initial_state or hardcode a solution in the goal '
        'test. Compute through state transitions. A dictionary literal reads the OLD state: '
        'new fields in it cannot read one another. Use sequential local assignments or separate '
        'stages for dependent calculations. Advance stage only when its arithmetic is computed. '
        'Goal tests must not fire before all requested arithmetic is complete. '
        'Use explicit units and conversions from the question; document needed conversion factors '
        'as named numeric state fields, not unexplained result constants. '
        'For arithmetic chains, use an integer stage and compute the next result from state '
        'facts at each transition. For constraints, enumerate feasible choices and test them. '
        'For minimum/maximum questions use objective="min"/"max" and enumerate ALL feasible '
        'goal states, answer returning their objective value. Otherwise objective="first". '
        f'The local BFS has depth {solver.cfg.max_depth} and {solver.cfg.max_expansions} expansions. '
        'Example FORMAT for a different problem, 2 bags of 5 beads then remove 3: '
        + json.dumps(example) + '. '
        'Use the actual problem facts, not the example numbers.')
    result = None
    status = 'model_error'
    expansions = 0
    model = None
    # Repair structural/runtime failures and unsuccessful searches, never reference-answer feedback.
    for repair in range(solver.cfg.retries + 1):
        model = solver.request(prompt, TOS_SCHEMA, 0.1, 'tos_model_synthesis', solver.cfg.max_tokens)
        if model is None:
            break
        attempt_events = []
        try:
            result, path, status, expansions = local_bfs(model, solver.cfg, attempt_events)
            solver.events.extend(attempt_events)
            solver.events.append({'kind': 'tos_search_attempt', 'repair': repair,
                                  'status': status, 'expansions': expansions, 'answer': result})
            if result is not None:
                if solver.cfg.audit_tos:
                    audit_prompt = (
                        f'Problem: {question}\nGenerated search model: {json.dumps(model)}\n'
                        f'Visited solution path: {json.dumps(path)}\nComputed answer: {result}.\n'
                        'Independently verify that the initial facts, number of stages, state '
                        'transitions, requested target and units match the question. Compute '
                        'the answer independently. Return valid=false if the model skips or '
                        'duplicates a stage, uses an unsupported assumption, or returns the '
                        'wrong quantity. Give the independent numeric answer and reason.')
                    audit = solver.request(audit_prompt, REVIEW_SCHEMA, 0.1,
                                           'tos_semantic_audit', solver.cfg.final_tokens)
                    accepted = (audit is not None and audit['valid'] and
                                math.isclose(audit['answer'], result, rel_tol=0, abs_tol=1e-4))
                    solver.events.append({'kind': 'tos_audit', 'review': audit,
                                          'computed_answer': result, 'accepted': accepted})
                    if not accepted:
                        solver.failure = 'tos_semantic_audit_rejected'
                        result = None
                        status = 'semantic_unverified'
                        prompt += ('\nPrevious model: ' + json.dumps(model) +
                                   '\nIndependent audit rejected the model: ' +
                                   str(audit.get('reason') if audit else 'no valid audit') +
                                   '. Repair the transitions and target without using a '
                                   'guessed final answer.')
                        continue
                break
            solver.failure = 'search_' + status
            feedback = ('Search ended with ' + status + '. Check successor progress, goal reachability, '
                        'and whether unrelated counters consume the depth budget. Do not change the '
                        'problem to fit a guessed answer. Use aggregate arithmetic instead of '
                        'enumerating time steps if the stated depth budget is insufficient.')
            prompt += '\nPrevious model: ' + json.dumps(model) + '\nLocal failure: ' + feedback
            prompt += '\nLast visited states: ' + json.dumps(attempt_events[-3:])
        except (ValueError, TypeError, KeyError, SyntaxError, ArithmeticError, IndexError,
                RecursionError) as exc:
            solver.events.extend(attempt_events)
            solver.failure = str(exc)
            status = 'model_error'
            solver.events.append({'kind': 'tos_model_rejected', 'error': str(exc), 'repair': repair})
            prompt += '\nPrevious model: ' + json.dumps(model) + '\nLocal failure: ' + str(exc)
            prompt += '. Repair the model while preserving the problem semantics.'
    if solver.calls >= solver.cfg.max_calls and result is None:
        status = 'call_limit'
    return {'method': TOS_METHOD, 'ans': result, 'status': status,
            'steps': path if result is not None else [], 'calls': solver.calls,
            'tokens': solver.tokens, 'sec': time.time()-started, 'trace': solver.events,
            'last_error': solver.failure, 'seed': solver.cfg.seed, 'config': vars(solver.cfg),
            'engine_version': '4', 'baseline_version': 'tos-arithmetic-3' if solver.cfg.audit_tos
            else 'tos-arithmetic-2',
            'search_model': model,
            'local_expansions': sum(event['kind'] == 'tos_expand' for event in solver.events),
            'adaptation': 'Per-problem restricted Python components; local BFS; no human review '
                          'or LLM calls within BFS; bounded Python interpreter; structural/runtime '
                          'and unsuccessful-search repair calls are counted.'}


## Ollama client


In [ ]:
"""Ollama transport for engine v3. Caching is disabled for measured runs."""
def ollama_backend(prompt, schema, temperature, seed, max_tokens):
    import ollama
    kwargs = {'model': Model, 'messages': [{'role': 'user', 'content': prompt}],
              'options': {'temperature': temperature, 'seed': seed, 'num_predict': max_tokens}}
    if schema is not None:
        kwargs['format'] = schema
    response = ollama.chat(**kwargs)
    return {'text': response['message']['content'],
            'prompt_tokens': response.get('prompt_eval_count', 0),
            'completion_tokens': response.get('eval_count', 0),
            'done_reason': response.get('done_reason', 'unknown')}


## Benchmark problems


In [ ]:
"""
Benchmark test dataset for Cost-Aware Heuristic Search for LLM Reasoning.
Includes:
- STANDARD_PROBLEMS: Multi-step elementary math problems (GSM8K style)
- HARD_PROBLEMS: Multi-step reasoning problems with constraint optimization, rates, or systems of equations
"""

STANDARD_PROBLEMS = [
    ("Sarah has 3 boxes of apples with 12 apples each. She sells 15 apples. How many are left?", 21),
    ((
        'A baker makes 8 trays of 6 cupcakes each. He sells 30 cupcakes. How many '
        'cupcakes does he have left?'
    ), 18),
    ("Tom has $50. He buys 4 notebooks that cost $6 each. How much money does he have left?", 26),
    ((
        'A school has 5 classrooms with 24 students in each. If 18 students are absent '
        'today, how many students are present?'
    ), 102),
    ("Maria reads 15 pages of a book every day. After 6 days, how many pages has she read?", 90),
    ((
        'A farmer has 7 rows of corn with 20 plants in each row. If 25 plants die, how '
        'many plants are left?'
    ), 115),
    ((
        'A store had 200 shirts. It sold 45 shirts on Monday and 38 shirts on Tuesday. '
        'How many shirts are left?'
    ), 117),
    ((
        'Jake has 4 packs of pencils with 10 pencils each. He gives 12 pencils to his '
        'friends. How many pencils does he have left?'
    ), 28),
    ((
        'A parking lot has 9 rows with 15 cars in each row. If 40 cars leave, how many '
        'cars remain?'
    ), 95),
    ((
        'A library has 6 shelves with 35 books on each shelf. If 90 books are checked '
        'out, how many books remain on the shelves?'
    ), 120),
]

HARD_PROBLEMS = [
    ((
        'A school is organizing a field trip. There are 4 buses, each holding 45 '
        'students. 172 students signed up, but 15 of them later cancelled. The school '
        'also needs to send 1 teacher for every 12 students who attend, rounding up. How '
        'many teachers are needed?'
    ), 14),
    ((
        'Kushal wants two bags of milk. He can convert 30gm of protein into 1 bag of '
        'milk. One lassi contains 10gm of protein. How many lassi does Kushal need to '
        'drink?'
    ), 6),
    ((
        'A tank is filled by pipe A at 20 liters per minute and drained by pipe B at 8 '
        'liters per minute, both running at the same time. If the tank starts empty and '
        'needs 600 liters to be full, how many minutes will it take?'
    ), 50),
    ((
        'A shop sells pens in packs of 6 for $9, and separately as single pens for $2 '
        'each. Priya needs exactly 20 pens and wants to spend as little money as '
        'possible. How much does she spend?'
    ), 31),
    ((
        'A train travels 60 km in the first hour, then its speed increases by 10 km/h '
        'every following hour. How far has it traveled after 4 hours in total?'
    ), 300),
    ((
        'You have 30 coins in dimes and quarters. Their total value is $5.10. Find the '
        'number of dimes you have.'
    ), 16),
]

PROBLEMS = STANDARD_PROBLEMS


## Official GSM8K and SVAMP: fixed development and evaluation subsets


In [ ]:
"""Official GSM8K/SVAMP snapshots and deterministic, disjoint benchmark subsets.

Sampling depends only on IDs and a fixed selection seed, never model outcomes.
Cache hashes and selected records are exported so results can be reproduced.
"""
import argparse
import hashlib
import json
import math
from pathlib import Path
import re
import urllib.request

SOURCES = {
    'gsm8k_train': {
        'url': 'https://raw.githubusercontent.com/openai/grade-school-math/master/'
               'grade_school_math/data/train.jsonl',
        'dataset': 'gsm8k', 'source_split': 'train', 'format': 'jsonl', 'count': 7473,
        'reference': 'https://arxiv.org/abs/2110.14168',
        'license': 'MIT (see upstream repository)',
    },
    'gsm8k_test': {
        'url': 'https://raw.githubusercontent.com/openai/grade-school-math/master/'
               'grade_school_math/data/test.jsonl',
        'dataset': 'gsm8k', 'source_split': 'test', 'format': 'jsonl', 'count': 1319,
        'reference': 'https://arxiv.org/abs/2110.14168',
        'license': 'MIT (see upstream repository)',
    },
    'svamp': {
        'url': 'https://raw.githubusercontent.com/arkilpatel/SVAMP/main/SVAMP.json',
        'dataset': 'svamp', 'source_split': 'challenge', 'format': 'json', 'count': 1000,
        'reference': 'https://arxiv.org/abs/2103.07191',
        'license': 'MIT (see upstream repository)',
    },
}
SELECTION_SEED = 203
# Freeze the verified upstream bytes, including when Kaggle downloads afresh.
SOURCE_SHA256 = {
    'gsm8k_train': '17f347dc51477c50d4efb83959dbb7c56297aba886e5544ee2aaed3024813465',
    'gsm8k_test': '3730d312f6e3440559ace48831e51066acaca737f6eabec99bccb9e4b3c39d14',
    'svamp': '5be77703a6d891ae476d7c082787ad361392aa02453b132516cdd5f4e7934e3e',
}


def parse_reference(value):
    """Reference-only parser, never applied to model output or sent to solvers."""
    if isinstance(value, bool):
        raise ValueError('boolean reference answer')
    text = str(value).strip()
    if '####' in text:
        text = text.rsplit('####', 1)[1].strip()
    text = text.replace(',', '').replace('$', '').strip()
    if not re.fullmatch(r'[+-]?(?:\d+(?:\.\d+)?|\.\d+)', text):
        raise ValueError('unsupported reference answer: ' + text)
    answer = float(text)
    if not math.isfinite(answer):
        raise ValueError('nonfinite reference answer')
    return answer


def fingerprint(question):
    normalized = ' '.join(question.lower().split())
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()


def normalize_records(source, records):
    spec = SOURCES[source]
    normalized = []
    identities = set()
    for index, row in enumerate(records):
        if spec['dataset'] == 'gsm8k':
            question = row['question'].strip()
            expected = parse_reference(row['answer'])
            original_id = str(index)
        else:
            question = (row['Body'].strip() + ' ' + row['Question'].strip()).strip()
            expected = parse_reference(row['Answer'])
            original_id = str(row['ID'])
        source_id = f'{spec["dataset"]}:{spec["source_split"]}:{original_id}'
        if not question or source_id in identities:
            raise ValueError('empty question or duplicate source ID')
        identities.add(source_id)
        normalized.append(dict(question=question, expected=expected,
                               dataset=spec['dataset'], source_split=spec['source_split'],
                               source_id=source_id, question_sha256=fingerprint(question)))
    return normalized


def read_snapshot(source, cache_dir, download=True):
    spec = SOURCES[source]
    cache = Path(cache_dir)
    cache.mkdir(parents=True, exist_ok=True)
    path = cache / (source + '.' + spec['format'])
    metadata_path = cache / (source + '.metadata.json')
    if not path.exists():
        if not download:
            raise FileNotFoundError('Missing dataset snapshot: ' + str(path))
        if metadata_path.exists():
            raise ValueError('Snapshot missing but provenance exists; restore it or use a new cache directory')
        request = urllib.request.Request(spec['url'], headers={'User-Agent': 'CostAwareBenchmark/3.1'})
        with urllib.request.urlopen(request, timeout=60) as response:
            raw = response.read(20_000_001)
        if len(raw) > 20_000_000:
            raise ValueError('unexpectedly large dataset download')
        if hashlib.sha256(raw).hexdigest() != SOURCE_SHA256[source]:
            raise ValueError('Upstream dataset changed; review it before changing the frozen source hash')
        records = (json.loads(raw) if spec['format'] == 'json' else
                   [json.loads(line) for line in raw.decode('utf-8').splitlines() if line.strip()])
        # Validate before caching a potentially truncated or wrong download.
        normalize_records(source, records)
        if len(records) != spec['count']:
            raise ValueError('Unexpected upstream dataset size for ' + source)
        metadata = dict(spec, sha256=hashlib.sha256(raw).hexdigest(), rows=len(records))
        with path.open('xb') as handle:
            handle.write(raw)
        with metadata_path.open('x', encoding='utf-8') as handle:
            json.dump(metadata, handle, indent=2)
    raw = path.read_bytes()
    metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    actual_hash = hashlib.sha256(raw).hexdigest()
    if (metadata['sha256'] != actual_hash or actual_hash != SOURCE_SHA256[source]
            or metadata['url'] != spec['url']):
        raise ValueError('Dataset snapshot provenance/hash mismatch: ' + str(path))
    records = (json.loads(raw) if spec['format'] == 'json' else
               [json.loads(line) for line in raw.decode('utf-8').splitlines() if line.strip()])
    if len(records) != spec['count']:
        raise ValueError('Unexpected cached dataset size')
    return normalize_records(source, records), metadata


def fixed_sample(records, count, seed, excluded=()):
    if count < 0:
        raise ValueError('sample count must be nonnegative')
    used = set(excluded)
    ordered = sorted(records, key=lambda row: hashlib.sha256(
        f'{seed}:{row["source_id"]}'.encode('utf-8')).hexdigest())
    result = []
    for row in ordered:
        if row['question_sha256'] not in used and len(result) < count:
            used.add(row['question_sha256'])
            result.append(dict(row))
    if len(result) != count:
        raise ValueError(f'Requested {count} distinct problems, only {len(result)} available')
    return result


def build_subsets(catalog, selection_seed=SELECTION_SEED, dev_count=30, gsm_count=100, svamp_count=50):
    # Exclude all GSM8K test questions from development, not just the sampled test subset.
    test_fingerprints = {row['question_sha256'] for row in catalog['gsm8k_test']}
    development = fixed_sample(catalog['gsm8k_train'], dev_count, selection_seed, test_fingerprints)
    used = {row['question_sha256'] for row in development}
    gsm = fixed_sample(catalog['gsm8k_test'], gsm_count, selection_seed, used)
    used.update(row['question_sha256'] for row in gsm)
    svamp = fixed_sample(catalog['svamp'], svamp_count, selection_seed, used)
    for row in development:
        row['evaluation_split'] = 'development'
    for row in gsm + svamp:
        row['evaluation_split'] = 'evaluation'
    return {'development': development, 'evaluation': gsm + svamp,
            'pilot': development[:min(20, len(development))]}


def prepare_benchmarks(cache_dir='kaggle/data/official', selection_seed=SELECTION_SEED, download=True):
    catalog, provenance = {}, {}
    for source in SOURCES:
        catalog[source], provenance[source] = read_snapshot(source, cache_dir, download)
    subsets = build_subsets(catalog, selection_seed)
    return subsets, {'selection_seed': selection_seed, 'sources': provenance,
                     'sampling': 'SHA256 rank of selection seed and source ID; exact question deduplication',
                     'development': '30 GSM8K train; excludes all exact GSM8K test matches',
                     'evaluation': '100 GSM8K test + 50 SVAMP challenge; no development overlap',
                     'pilot': 'first 20 selected development questions; not final evaluation'}


def export_subsets(output, subsets, provenance):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=False)
    for name, records in subsets.items():
        with (output / (name + '.jsonl')).open('x', encoding='utf-8') as handle:
            for record in records:
                handle.write(json.dumps(record, ensure_ascii=False) + '\n')
    (output / 'dataset_manifest.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')




## Metrics and charts


In [ ]:
"""Report one selected experiment; never pool old and repaired implementations."""
import argparse
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


def load_all_results(input_files):
    rows = []
    for filename in input_files:
        with open(filename, encoding='utf-8') as handle:
            rows.extend(json.loads(line) for line in handle if line.strip())
    if not rows:
        raise ValueError('No results in the selected files')
    versions = {row.get('engine_version', 'legacy') for row in rows}
    if len(versions) != 1:
        raise ValueError('Do not mix legacy and repaired engine results')
    if len({row.get('experiment_id', 'unspecified') for row in rows}) != 1:
        raise ValueError('Select a single experiment; do not pool different runs')
    tos_versions = {row.get('baseline_version', 'unknown') for row in rows
                    if str(row['method']).startswith('Thought of Search')}
    if len(tos_versions) > 1:
        raise ValueError('Do not mix different Thought of Search adaptation versions')
    df = pd.DataFrame(rows)
    if 'seed' not in df:
        df['seed'] = 'legacy'
    if 'dataset' not in df:
        df['dataset'] = 'custom'
    if df.duplicated(['dataset', 'method', 'seed', 'problem_id']).any():
        raise ValueError('Duplicate problem/method/seed records; select a single experiment')
    df['outcome'] = ['missing' if pd.isna(row.get('ans', row.get('final_answer'))) else
                     'correct' if row.get('correct') is True else 'wrong' for row in rows]
    df['correct'] = df['outcome'].eq('correct')
    return df


def generate_report_and_plots(output_dir='kaggle/results', input_files=None):
    if not input_files:
        raise ValueError('Select input_files explicitly so historical experiments are not pooled')
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    df = load_all_results(input_files)
    df['missing'] = df['outcome'].eq('missing')
    df['wrong'] = df['outcome'].eq('wrong')
    summary = df.groupby(['dataset', 'method']).agg(
        Runs=('correct', 'size'), Correct=('correct', 'sum'),
        Missing=('missing', 'sum'), Wrong=('wrong', 'sum'),
        Accuracy=('correct', lambda x: x.mean()*100),
        Mean_Calls=('calls', 'mean'), Mean_Tokens=('tokens', 'mean'),
        Avg_Time_Sec=('sec', 'mean')).reset_index()
    per_seed = df.groupby(['dataset', 'method', 'seed']).agg(
        Accuracy=('correct', lambda x: x.mean()*100), Mean_Calls=('calls', 'mean'),
        Mean_Tokens=('tokens', 'mean')).reset_index()
    spread = per_seed.groupby(['dataset', 'method']).agg(
        Seeds=('seed', 'size'), Accuracy_Seed_SD=('Accuracy', 'std')).reset_index()
    summary = summary.merge(spread, on=['dataset', 'method']).sort_values(['dataset', 'Accuracy'],
                                                                           ascending=[True, False])
    references = summary.loc[summary.method.eq('Tree-of-Thoughts (LLM-Judge)')].set_index('dataset').Mean_Calls
    if not references.empty:
        denominators = summary.dataset.map(references).replace(0, float('nan'))
        summary['Call_Savings_vs_ToT_%'] = (1-summary.Mean_Calls/denominators)*100
    summary.to_csv(output / 'summary_table.csv', index=False)
    per_seed.to_csv(output / 'per_seed_summary.csv', index=False)
    # Save failure counts and terminal statuses alongside accuracy.
    if 'status' in df:
        df.groupby(['dataset', 'method', 'status']).size().rename('count').to_csv(output / 'termination_counts.csv')
    print(summary.to_string(index=False))
    datasets = list(summary.dataset.unique())
    fig, axes = plt.subplots(len(datasets), 1, figsize=(12, 7*len(datasets)), squeeze=False)
    for dataset, ax in zip(datasets, axes[:, 0]):
        for _, row in summary.loc[summary.dataset.eq(dataset)].iterrows():
            ax.scatter(row.Mean_Calls, row.Accuracy, s=90)
            ax.annotate(row.method, (row.Mean_Calls, row.Accuracy), xytext=(6, 6),
                        textcoords='offset points', fontsize=8)
        ax.set(xlabel='Mean LLM calls (all retries and reviews included)', ylabel='Accuracy (%)',
               ylim=(-5, 105), title=dataset + ': accuracy and inference cost')
        ax.grid(alpha=.25)
    fig.tight_layout()
    fig.savefig(output / 'pareto_frontier.png', dpi=200)
    plt.close(fig)
    fig, axes = plt.subplots(len(datasets), 1, figsize=(12, 5*len(datasets)), squeeze=False)
    for dataset, ax in zip(datasets, axes[:, 0]):
        subset = summary.loc[summary.dataset.eq(dataset)]
        ax.barh(subset.method, subset.Mean_Tokens)
        ax.set(xlabel='Mean prompt + completion tokens', title=dataset + ': token consumption')
    fig.tight_layout()
    fig.savefig(output / 'token_consumption.png', dpi=200)
    plt.close(fig)
    return summary




## Benchmark runner


In [ ]:
"""Run seeded experiments and save every result immediately to a new directory."""
import argparse
import json
from datetime import datetime, timezone
from pathlib import Path



def run_suite(backend, output, seeds=(11, 23, 37), problems=None, methods=None,
              dataset_provenance=None, profile='custom'):
    methods = list(METHODS if methods is None else methods)
    if not methods or len(set(methods)) != len(methods) or any(method not in METHODS for method in methods):
        raise ValueError('Select distinct known benchmark methods')
    seeds = list(seeds)
    if not seeds or len(set(seeds)) != len(seeds):
        raise ValueError('Select distinct seeds')
    problems = problems if problems is not None else STANDARD_PROBLEMS + HARD_PROBLEMS
    cases = []
    for index, problem in enumerate(problems):
        case = dict(problem) if isinstance(problem, dict) else {
            'question': problem[0], 'expected': problem[1], 'dataset': 'custom',
            'source_id': f'custom:{index}', 'source_split': 'custom', 'evaluation_split': profile}
        if not isinstance(case.get('question'), str) or not case['question'].strip() or not numeric(case['expected']):
            raise ValueError('Each problem needs a question and finite numeric expected answer')
        case.setdefault('dataset', 'custom')
        case.setdefault('source_id', f'custom:{index}')
        case.setdefault('evaluation_split', profile)
        cases.append(case)
    if not cases or len({(case['dataset'], case['source_id']) for case in cases}) != len(cases):
        raise ValueError('Need nonempty problems with distinct dataset/source IDs')
    if len({case['evaluation_split'] for case in cases}) != 1:
        raise ValueError('Do not mix development and evaluation problems in one experiment')
    output = Path(output)
    output.mkdir(parents=True, exist_ok=False)
    manifest = {'engine_version': '3', 'seeds': seeds, 'methods': methods,
                'model': 'qwen2.5:7b', 'config': vars(Config()),
                'baseline_versions': {'thought_of_search': 'tos-arithmetic-2'},
                'problems': cases, 'profile': profile, 'dataset_provenance': dataset_provenance,
                'planned_solver_runs': len(cases)*len(methods)*len(seeds),
                'created_utc': datetime.now(timezone.utc).isoformat(),
                'policy': 'No cache; shared step generator, terminal review and budgets. '
                          'Greedy uses one candidate and beam=1. ToT alone judges every step. '
                          'Standard CoT uses one complete 4096-token request without review. '
                          'Thought of Search synthesizes a per-problem restricted Python model, '
                          'then local BFS with no LLM calls or terminal review during search. '
                          'All synthesis/repair calls count; no domain-level amortization. '
                          'ToS v2 uses bounded Python statements and repairs unsuccessful searches.',
                'references': {'thought_of_search': 'https://arxiv.org/abs/2404.11833'}}
    (output / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    with (output / 'selected_problems.jsonl').open('x', encoding='utf-8') as selected:
        for case in cases:
            selected.write(json.dumps(case, ensure_ascii=False) + '\n')
    rows = []
    with (output / 'full_results.jsonl').open('w', encoding='utf-8') as handle:
        for seed in seeds:
            for pid, case in enumerate(cases):
                for method in methods:
                    result = evaluate(backend, case['question'], case['expected'], method, Config(seed=seed), pid)
                    result['experiment_id'] = output.name
                    result['profile'] = profile
                    for key in ('dataset', 'source_id', 'source_split', 'evaluation_split', 'question_sha256'):
                        if key in case:
                            result[key] = case[key]
                    handle.write(json.dumps(result) + '\n')
                    handle.flush()
                    rows.append(result)
                    detail = ''
                    if result['outcome'] == 'missing':
                        detail = f' [{result["status"]}: {str(result.get("last_error") or "no goal reached")[:160]}]'
                    elif result['outcome'] == 'wrong':
                        detail = f' [answer={result["ans"]}, expected={result["expected"]}]'
                    print(f'Seed {seed}, {case["source_id"]}, {method}: '
                          f'{result["outcome"]} ({result["calls"]} calls){detail}', flush=True)
    generate_report_and_plots(str(output), input_files=[str(output / 'full_results.jsonl')])
    return rows




## Run the benchmark


In [ ]:
from datetime import datetime
PROFILE = 'pilot'  # 'pilot': 20 development questions; 'evaluation': 100 GSM8K + 50 SVAMP.
SEEDS = [11] if PROFILE != 'evaluation' else [11, 23, 37]
# None runs all six methods. Use [METHODS[5]] to run only the new ToS baseline.
METHODS_TO_RUN = None
OUTPUT = 'results_v4_' + datetime.now().strftime('%Y%m%d_%H%M%S')
subsets, data_provenance = prepare_benchmarks(cache_dir='benchmark_data_cache')
selected_problems = subsets[PROFILE]
print('Planned solver runs:', len(selected_problems)*len(SEEDS)*len(METHODS_TO_RUN or METHODS))
all_results = run_suite(ollama_backend, OUTPUT, seeds=SEEDS, methods=METHODS_TO_RUN,
                        problems=selected_problems, dataset_provenance=data_provenance, profile=PROFILE)


## Download results


In [ ]:
import shutil
archive = shutil.make_archive(OUTPUT, 'zip', OUTPUT)
print('Download:', archive)
